# Transport test, round 1: does a small fine-tune land on existing VPD subcomponents?

Results in `experiments/transport_test/results/`, figures in `experiments/transport_test/figures/`, scoring in `reports/2026-10-14_analysis_transport-test.md`. Run `python experiments/transport_test/plot.py` first if the figures are missing.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
R = Path('../experiments/transport_test/results'); F = Path('../experiments/transport_test/figures')
load = lambda n: json.loads((R / n).read_text())
cal, A, B, fA, fB = load('calibration.json'), load('transport_A.json'), load('transport_B.json'), load('faithfulness_A.json'), load('faithfulness_B.json')
print('calibration vs paper:', {k: round(v, 2) for k, v in cal['ratio_to_paper'].items()})

## Could it do it? How big were the fine-tunes?

Arm A reaches P(o) = 0.999 on held-out emoticon positions by step 10 with a 1% relative change in the largest matrix; arm B moves the model 0.01 to 0.02 nats. The step sweep did not span arm A's transition.

In [ ]:
display(Image(F / 'finetune_curves.png'))

## Could it do it? Is the update in the span of the existing subcomponents? (measure 1)

No. The Frobenius fraction of δW on the alive subcomponents is 1.5 to 1.7 times that of a random direction of the same norm, not the predicted 10 times (prediction 2, first half: reversed).

In [ ]:
display(Image(F / 'frobenius_fraction.png'))
s = A['steps'][str(A['scored_checkpoint'])]; sb = B['steps'][str(B['scored_checkpoint'])]
print('arm A alive/all/random:', round(s['T0_frobenius']['fraction_alive_pooled'], 5), round(s['T0_frobenius']['fraction_all_pooled'], 5), [round(x, 5) for x in s['arm_N_fraction_alive_pooled']])
print('arm B alive/all/random:', round(sb['T0_frobenius']['fraction_alive_pooled'], 5), round(sb['T0_frobenius']['fraction_all_pooled'], 5), [round(x, 5) for x in sb['arm_N_fraction_alive_pooled']])

## Does it match the prediction? Functional capture against k, top-k versus random (measure 2)

Arm A: ten edited subcomponents reproduce the emoticon behaviour, and so do ten random ones (prediction 1a: margin 0.013, inside the null band; reversed). Only k = 3 separates them, by 0.16. Arm B: every fit is worse than doing nothing on held-out rows (prediction 2, second half: reversed); training-set capture was not recorded, so overfitting versus under-parametrisation is open.

In [ ]:
display(Image(F / 'capture_vs_k_armA.png')); display(Image(F / 'capture_vs_k_armB.png'))
for k in ('1', '3', '10', '30', '100'):
    r = s['T1'][k]
    print(f"arm A k={k:>3}: top {r['write']['primary']['capture']:.3f}  random {[round(x['primary']['capture'], 3) for x in r.get('random_write', [])]}  read {r.get('read', {}).get('primary', {}).get('capture', float('nan')):.3f}")

## Where did the update go? (measure 4, prediction 1b, held)

Among the alive subcomponents of `h.2.mlp.down_proj`, the paper's emoticon subcomponent 2359 ranks second by Frobenius contribution at step 10, with three siblings in the top eight; by step 30 the family has dispersed.

In [ ]:
for st in sorted(A['steps'], key=int):
    print('step', st, A['steps'][st]['1b_location']['ranks'])

## Is the transported decomposition still faithful? (measure 3, prediction 3, held)

Stochastic ratios to the released decomposition on W0 are 0.94 to 1.02 everywhere. Adversarial ratios pass for the scored variants (1.43 for arm A, 1.05 for arm B); the null transport at arm A's step 10 reaches 2.4, driven by one of the two row blocks, and is 1.0 to 1.3 at later checkpoints.

In [ ]:
display(Image(F / 'faithfulness_ratios.png'))
for arm, f in (('A', fA), ('B', fB)):
    for st in sorted(f['steps'], key=int):
        print(arm, st, {v: {m: round(x, 2) for m, x in d['ratio_to_calibration'].items()} for v, d in f['steps'][st].items()})

## Summary

The literal projection of δW onto the subcomponent dictionary explains neither the parameters nor the behaviour of these updates. The behaviour of the emoticon fine-tune is easy to mimic with a few edited subcomponents, but the null used here (free write vectors on random subcomponents) is expressive enough to mimic it too, so round 2 needs a scale-only or constrained null and a functional selector. The decomposition does not break: faithfulness holds under every transport by the stochastic measure. Next steps are listed in `backlog.md` and go through `/design`.